In [2]:
!pip install xgboost lightgbm catboost prophet optuna scipy shap -q
import pandas as pd
import numpy as np
import warnings
import optuna
from scipy.optimize import minimize_scalar

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from prophet import Prophet
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

# ── Cấu hình ──────────────────────────────────────────────────────────────────
RANDOM_SEED  = 42
DIR          = '../dataset/'          # Thư mục chứa dữ liệu
OUTPUT_FILE  = 'submission_final.csv'
N_TRIALS     = 30                   # Số Optuna trials mỗi model

np.random.seed(RANDOM_SEED)

# ==============================================================================
# 1. TẢI DỮ LIỆU
# ==============================================================================
print("=" * 65)
print("STEP 1 — Tải dữ liệu")
print("=" * 65)

df_sales  = pd.read_csv(DIR + 'sales.csv',             parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df_sub    = pd.read_csv(DIR + 'sample_submission.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df_promos = pd.read_csv(DIR + 'promotions.csv',        parse_dates=['start_date', 'end_date'])

print(f"Train : {df_sales.shape}  | {df_sales.Date.min().date()} → {df_sales.Date.max().date()}")
print(f"Test  : {df_sub.shape}    | {df_sub.Date.min().date()} → {df_sub.Date.max().date()}")

df_sales['Revenue_log'] = np.log1p(df_sales['Revenue'])
df_sales['COGS_log']    = np.log1p(df_sales['COGS'])
df_sales['COGS_Ratio']  = df_sales['COGS'] / (df_sales['Revenue'] + 1e-9)
MIN_DATE = df_sales['Date'].min()

# ==============================================================================
# 2. PROMO PROBABILITY
# ==============================================================================
promo_dates = []
for _, row in df_promos.iterrows():
    promo_dates.extend(pd.date_range(row['start_date'], row['end_date']))
promo_df = pd.DataFrame({'Date': promo_dates})
promo_df['month'] = promo_df['Date'].dt.month
promo_df['day']   = promo_df['Date'].dt.day
promo_counts = promo_df.groupby(['month', 'day']).size().reset_index(name='promo_years')
promo_counts['promo_prob'] = (promo_counts['promo_years'] / 10.0).clip(upper=1.0)

# ==============================================================================
# 3. SALE SEASONS
# ==============================================================================
SALE_SEASONS = [
    {'month': 1,  'start_day': 30, 'duration': 31, 'profit_rank': 1},
    {'month': 3,  'start_day': 18, 'duration': 31, 'profit_rank': 2},
    {'month': 6,  'start_day': 23, 'duration': 30, 'profit_rank': 3},
    {'month': 7,  'start_day': 30, 'duration': 35, 'profit_rank': 5},
    {'month': 8,  'start_day': 30, 'duration': 33, 'profit_rank': 4},
    {'month': 11, 'start_day': 18, 'duration': 45, 'profit_rank': 6},
]

def _get_sale_windows(years):
    windows = []
    for year in years:
        for s in SALE_SEASONS:
            try:
                start = pd.Timestamp(year=year, month=s['month'], day=s['start_day'])
            except ValueError:
                start = pd.Timestamp(year=year, month=s['month'], day=1) + pd.offsets.MonthEnd(0)
            end = start + pd.Timedelta(days=s['duration'] - 1)
            windows.append((start, end, s['profit_rank']))
    return windows

def add_sale_features(df):
    df = df.copy()
    windows = _get_sale_windows(df['Date'].dt.year.unique())
    df['is_sale_season']       = 0
    df['sale_rank']            = 0
    df['sale_progress']        = 0.0
    df['days_to_next_sale']    = 999.0
    df['days_since_last_sale'] = 999.0
    for start, end, rank in windows:
        m_in     = (df['Date'] >= start) & (df['Date'] <= end)
        m_before = df['Date'] < start
        m_after  = df['Date'] > end
        df.loc[m_in, 'is_sale_season'] = 1
        df.loc[m_in, 'sale_rank']      = rank
        duration = (end - start).days + 1
        df.loc[m_in, 'sale_progress']  = (df.loc[m_in, 'Date'] - start).dt.days / duration
        d2n = (start - df.loc[m_before, 'Date']).dt.days
        df.loc[m_before, 'days_to_next_sale'] = np.minimum(df.loc[m_before, 'days_to_next_sale'], d2n)
        d2l = (df.loc[m_after, 'Date'] - end).dt.days
        df.loc[m_after, 'days_since_last_sale'] = np.minimum(df.loc[m_after, 'days_since_last_sale'], d2l)
    df.loc[df['is_sale_season'] == 1, 'days_to_next_sale']    = 0
    df.loc[df['is_sale_season'] == 1, 'days_since_last_sale'] = 0
    return df

def sale_density_30(df):
    """% ngày sale trong 30 ngày tới — calendar-only, không leak."""
    windows = _get_sale_windows(
        df['Date'].dt.year.unique().tolist() + (df['Date'].dt.year + 1).unique().tolist()
    )
    density = np.zeros(len(df))
    for i, d in enumerate(df['Date']):
        future_30 = pd.date_range(d + pd.Timedelta(days=1), periods=30)
        count = sum(1 for fd in future_30 for (s, e, _) in windows if s <= fd <= e)
        density[i] = count / 30.0
    return density

def days_to_nearest_peak(date):
    """Khoảng cách đến peak tháng 4, 5, 11."""
    y = date.year
    candidates = [
        pd.Timestamp(y-1, 4, 1), pd.Timestamp(y-1, 5, 1), pd.Timestamp(y-1, 11, 1),
        pd.Timestamp(y,   4, 1), pd.Timestamp(y,   5, 1), pd.Timestamp(y,   11, 1),
        pd.Timestamp(y+1, 4, 1), pd.Timestamp(y+1, 5, 1), pd.Timestamp(y+1, 11, 1),
    ]
    return min(abs((date - c).days) for c in candidates)

# ==============================================================================
# 4. PROPHET
# ==============================================================================
def fit_prophet(dates, log_revenues):
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0,
        changepoint_range=0.95,
        seasonality_mode='additive',
    )
    m.fit(pd.DataFrame({'ds': dates, 'y': log_revenues}))
    return m

def prophet_components(model, dates):
    """Trả về yhat + 4 internal components làm features."""
    fc = model.predict(pd.DataFrame({'ds': dates}))
    return {
        'yhat'          : fc['yhat'].values,
        'prophet_trend' : fc['trend'].values,
        'prophet_weekly': fc['weekly'].values,
        'prophet_yearly': fc['yearly'].values,
        'prophet_uncert': (fc['yhat_upper'] - fc['yhat_lower']).values,
    }

# ==============================================================================
# 5. BUILD STATIC FEATURES
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 2 — Build Static Features")
print("=" * 65)

def build_static_features(df):
    out = df[['Date']].copy().sort_values('Date').reset_index(drop=True)
    yr  = out['Date'].dt.year
    mo  = out['Date'].dt.month
    dy  = out['Date'].dt.day
    dow = out['Date'].dt.dayofweek
    dim = out['Date'].dt.days_in_month

    # ── Calendar ──────────────────────────────────────────────────────────
    out['year']          = yr
    out['month']         = mo
    out['day']           = dy
    out['dow']           = dow
    out['is_weekend']    = (dow >= 5).astype(int)
    out['is_monthend']   = out['Date'].dt.is_month_end.astype(int)
    out['is_monthstart'] = out['Date'].dt.is_month_start.astype(int)
    out['t']             = (out['Date'] - MIN_DATE).dt.days   # linear time index
    out['doy_norm']      = out['Date'].dt.dayofyear / 365.0   # 0→1 trong năm
    out['month_progress']= dy / dim                            # 0→1 trong tháng
    out['year_parity']   = (yr % 2).astype(int)               # odd=1/even=0 (campaign cycle)

    # ── Fourier encoding (monthly) ─────────────────────────────────────────
    out['month_sin'] = np.sin(2 * np.pi * mo / 12)
    out['month_cos'] = np.cos(2 * np.pi * mo / 12)

    # ── Payday (cuối tháng 28-31 hoặc đầu tháng 1-3) ───────────
    _dy         = dy.values
    dist_start  = np.where(_dy <= 3,  0, (_dy - 3).astype(float))
    dist_end    = np.where(_dy >= 28, 0, (28 - _dy).astype(float))
    _d2p        = np.minimum(dist_start, dist_end)
    out['is_payday']              = (_d2p == 0).astype(int)
    out['days_to_nearest_payday'] = _d2p
    out['payday_proximity']       = np.exp(-_d2p / 3)   # decay exp, max ~12 ngày

    # ── Peak season (tháng 4, 5, 11) ─────────────────────────────
    out['is_peak_season'] = mo.isin([4, 5, 11]).astype(int)
    out['peak_proximity'] = 1 / (1 + out['Date'].map(days_to_nearest_peak))

    # ── Sale features ──────────────────────────────────────────────────────
    sale_df = add_sale_features(out[['Date']])
    for col in ['is_sale_season', 'sale_rank', 'sale_progress',
                'days_to_next_sale', 'days_since_last_sale']:
        out[col] = sale_df[col].values
    out['anticipation_signal'] = np.exp(-out['days_to_next_sale']    / 14)
    out['hangover_signal']     = np.exp(-out['days_since_last_sale'] / 14)

    # ── Sale density (% ngày sale trong 30 ngày tới) ──────────────────────
    print(f"  Computing sale_density_30 for {len(out)} rows...")
    out['sale_density_30'] = sale_density_30(out)

    # ── Promo probability (từ promotions.csv) ─────────────────────────────
    out = out.merge(promo_counts[['month', 'day', 'promo_prob']],
                    on=['month', 'day'], how='left')
    out['promo_prob'] = out['promo_prob'].fillna(0)

    return out.sort_values('Date').reset_index(drop=True)

print("Building train static features...")
train_static = build_static_features(df_sales)
print("Building test static features...")
test_static  = build_static_features(df_sub)

STATIC_COLS = [c for c in train_static.columns if c != 'Date']

def add_interactions(df):
    """Interaction terms — thêm SAU khi đã có prophet_uncert."""
    df = df.copy()
    df['sale_x_peak']   = df['sale_rank']       * df['is_peak_season']
    df['promo_x_sale']  = df['promo_prob']       * df['is_sale_season']
    df['uncert_x_sale'] = df['prophet_uncert']   * df['is_sale_season']
    df['payday_x_sale'] = df['payday_proximity'] * df['is_sale_season']
    return df

PROPHET_COLS  = ['prophet_trend', 'prophet_weekly', 'prophet_yearly', 'prophet_uncert']
INTERACT_COLS = ['sale_x_peak', 'promo_x_sale', 'uncert_x_sale', 'payday_x_sale']
ALL_FEAT_COLS = STATIC_COLS + PROPHET_COLS + INTERACT_COLS

print(f"\nTotal features: {len(ALL_FEAT_COLS)}  "
      f"(static={len(STATIC_COLS)}, prophet={len(PROPHET_COLS)}, interact={len(INTERACT_COLS)})")

# ==============================================================================
# 6. PRE-COMPUTE PROPHET CHO MỌI FOLD
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 3 — Pre-compute Prophet")
print("=" * 65)

FOLDS = [
    ('2012-07-04', '2018-12-31', '2019-01-01', '2019-12-31'),
    ('2012-07-04', '2019-12-31', '2020-01-01', '2020-12-31'),
    ('2012-07-04', '2020-12-31', '2021-01-01', '2021-12-31'),
    ('2012-07-04', '2021-12-31', '2022-01-01', '2022-12-31'),
]

def make_X_from(static_df, dates, comp):
    sf = static_df[static_df['Date'].isin(dates)].copy()
    sf = sf.set_index('Date').loc[dates.values].reset_index()
    for col in PROPHET_COLS:
        sf[col] = comp[col]
    sf = add_interactions(sf)
    sf[ALL_FEAT_COLS] = sf[ALL_FEAT_COLS].fillna(-1)
    return sf[ALL_FEAT_COLS].values

fold_cache = []

for i, (tr_start, tr_end, val_start, val_end) in enumerate(FOLDS):
    print(f"  Fold {i+1}: Prophet fit {tr_start}→{tr_end} ...")
    tr_mask  = (df_sales['Date'] >= tr_start) & (df_sales['Date'] <= tr_end)
    val_mask = (df_sales['Date'] >= val_start) & (df_sales['Date'] <= val_end)
    df_tr    = df_sales[tr_mask].reset_index(drop=True)
    df_val   = df_sales[val_mask].reset_index(drop=True)

    p        = fit_prophet(df_tr['Date'], df_tr['Revenue_log'])
    comp_tr  = prophet_components(p, df_tr['Date'])
    comp_val = prophet_components(p, df_val['Date'])

    fold_cache.append({
        'X_tr'          : make_X_from(train_static, df_tr['Date'],  comp_tr),
        'X_val'         : make_X_from(train_static, df_val['Date'], comp_val),
        'y_tr_rev'      : df_tr['Revenue_log'].values  - comp_tr['yhat'],
        'y_val_rev'     : df_val['Revenue_log'].values - comp_val['yhat'],
        'val_trend'     : comp_val['yhat'],
        'y_val_true'    : df_val['Revenue'].values,
        'y_tr_rat'      : df_tr['COGS_Ratio'].values,
        'y_val_rat'     : df_val['COGS_Ratio'].values,
        'y_tr_cog'      : df_tr['COGS_log'].values,
        'y_val_cog'     : df_val['COGS_log'].values,
        'y_val_cog_true': df_val['COGS'].values,
        'val_dates'     : df_val['Date'].values,
    })
    print(f"    → {len(df_tr):,} train / {len(df_val):,} val rows")

TUNE_FOLDS = [fold_cache[2], fold_cache[3]]   # 2 fold gần nhất cho Optuna

# ==============================================================================
# 7. OPTUNA — TỰ ĐỘNG TÌM THAM SỐ TỐI ƯU
# ==============================================================================
print("\n" + "=" * 65)
print(f"STEP 4 — Optuna Tuning ({N_TRIALS} trials × 3 models)")
print("=" * 65)

def _cv_mae(model_fn, tune_folds):
    """Mean Revenue MAE trên tune_folds."""
    maes = []
    for fd in tune_folds:
        m = model_fn()
        m.fit(fd['X_tr'], fd['y_tr_rev'],
              eval_set=[(fd['X_val'], fd['y_val_rev'])],
              **({'callbacks': [lgb.early_stopping(40, verbose=False)]}
                 if isinstance(m, lgb.LGBMRegressor) else {'verbose': False}))
        pred_log = fd['val_trend'] + m.predict(fd['X_val'])
        maes.append(mean_absolute_error(fd['y_val_true'], np.expm1(pred_log)))
    return float(np.mean(maes))

# ── XGBoost ───────────────────────────────────────────────────────────────────
def obj_xgb(trial):
    p = dict(
        n_estimators=2000, tree_method='hist', random_state=RANDOM_SEED,
        objective='reg:pseudohubererror',
        learning_rate    = trial.suggest_float('lr',    0.01, 0.15, log=True),
        max_depth        = trial.suggest_int('depth',   3, 7),
        subsample        = trial.suggest_float('sub',   0.5, 1.0),
        colsample_bytree = trial.suggest_float('col',   0.5, 1.0),
        min_child_weight = trial.suggest_int('mcw',     1, 15),
        reg_alpha        = trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('lam',   1e-4, 10.0, log=True),
    )
    return _cv_mae(lambda: xgb.XGBRegressor(**p, early_stopping_rounds=40), TUNE_FOLDS)

print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_xgb.optimize(obj_xgb, n_trials=N_TRIALS, show_progress_bar=True)
p = study_xgb.best_params
XGB_BEST = dict(n_estimators=2000, tree_method='hist', random_state=RANDOM_SEED,
                objective='reg:pseudohubererror',
                learning_rate=p['lr'], max_depth=p['depth'], subsample=p['sub'],
                colsample_bytree=p['col'], min_child_weight=p['mcw'],
                reg_alpha=p['alpha'], reg_lambda=p['lam'])
print(f"  Best XGB MAE: {study_xgb.best_value:,.0f}  lr={p['lr']:.4f} depth={p['depth']}")

# ── LightGBM ──────────────────────────────────────────────────────────────────
def obj_lgb(trial):
    p = dict(
        n_estimators=2000, objective='huber', random_state=RANDOM_SEED, verbose=-1,
        learning_rate    = trial.suggest_float('lr',    0.005, 0.1,   log=True),
        max_depth        = trial.suggest_int('depth',   3, 7),
        num_leaves       = trial.suggest_int('leaves',  15, 100),
        subsample        = trial.suggest_float('sub',   0.5, 1.0),
        colsample_bytree = trial.suggest_float('col',   0.5, 1.0),
        reg_alpha        = trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('lam',   1e-4, 10.0, log=True),
        min_child_samples= trial.suggest_int('mcs',     5, 50),
    )
    def make():
        m = lgb.LGBMRegressor(**p)
        m._early = True
        return m
    return _cv_mae(lambda: lgb.LGBMRegressor(**p), TUNE_FOLDS)

print("Tuning LightGBM...")
study_lgb = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_lgb.optimize(obj_lgb, n_trials=N_TRIALS, show_progress_bar=True)
p = study_lgb.best_params
LGB_BEST = dict(n_estimators=2000, objective='huber', random_state=RANDOM_SEED, verbose=-1,
                learning_rate=p['lr'], max_depth=p['depth'], num_leaves=p['leaves'],
                subsample=p['sub'], colsample_bytree=p['col'],
                reg_alpha=p['alpha'], reg_lambda=p['lam'], min_child_samples=p['mcs'])
print(f"  Best LGB MAE: {study_lgb.best_value:,.0f}  lr={p['lr']:.4f} depth={p['depth']}")

# ── CatBoost ──────────────────────────────────────────────────────────────────
def obj_cat(trial):
    p = dict(
        iterations=2000, loss_function='Huber:delta=1.5',
        random_state=RANDOM_SEED, verbose=False,
        learning_rate       = trial.suggest_float('lr',  0.01, 0.15, log=True),
        depth               = trial.suggest_int('depth', 3, 8),
        l2_leaf_reg         = trial.suggest_float('l2',  1.0, 20.0, log=True),
        bagging_temperature = trial.suggest_float('bt',  0.0, 1.0),
    )
    class _W:
        def __init__(self):  self.m = CatBoostRegressor(**p, early_stopping_rounds=40)
        def fit(self, X, y, eval_set, **kw): self.m.fit(X, y, eval_set=eval_set, verbose=False)
        def predict(self, X): return self.m.predict(X)
    return _cv_mae(lambda: _W(), TUNE_FOLDS)

print("Tuning CatBoost...")
study_cat = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_cat.optimize(obj_cat, n_trials=N_TRIALS, show_progress_bar=True)
p = study_cat.best_params
CAT_BEST = dict(iterations=2000, loss_function='Huber:delta=1.5',
                random_state=RANDOM_SEED, verbose=False,
                learning_rate=p['lr'], depth=p['depth'],
                l2_leaf_reg=p['l2'], bagging_temperature=p['bt'])
print(f"  Best CAT MAE: {study_cat.best_value:,.0f}  lr={p['lr']:.4f} depth={p['depth']}")

# ==============================================================================
# 8. WALK-FORWARD CV — thu thập OOF predictions
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 5 — Walk-Forward CV (OOF collection)")
print("=" * 65)

oof_rev_xgb, oof_rev_lgb, oof_rev_cat = [], [], []
oof_rat_xgb, oof_rat_lgb, oof_rat_cat = [], [], []
oof_cog_xgb, oof_cog_lgb, oof_cog_cat = [], [], []
oof_y_rev_log, oof_y_rat, oof_y_cog_log = [], [], []
oof_y_rev_true, oof_y_cog_true          = [], []
oof_dates_all = []

for i, fd in enumerate(fold_cache):
    print(f"\nFold {i+1}: {FOLDS[i][0]}→{FOLDS[i][1]} | val {FOLDS[i][2]}→{FOLDS[i][3]}")
    X_tr, X_val = fd['X_tr'], fd['X_val']

    for target_name, y_tr, y_val, oof_xgb, oof_lgb, oof_cat, oof_y in [
        ('Revenue',    fd['y_tr_rev'], fd['y_val_rev'], oof_rev_xgb, oof_rev_lgb, oof_rev_cat, oof_y_rev_log),
        ('COGS_Ratio', fd['y_tr_rat'], fd['y_val_rat'], oof_rat_xgb, oof_rat_lgb, oof_rat_cat, oof_y_rat),
        ('COGS_log',   fd['y_tr_cog'], fd['y_val_cog'], oof_cog_xgb, oof_cog_lgb, oof_cog_cat, oof_y_cog_log),
    ]:
        mx = xgb.XGBRegressor(**XGB_BEST, early_stopping_rounds=50)
        mx.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        ml = lgb.LGBMRegressor(**LGB_BEST)
        ml.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
               callbacks=[lgb.early_stopping(50, verbose=False)])

        mc = CatBoostRegressor(**CAT_BEST, early_stopping_rounds=50)
        mc.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        if target_name == 'Revenue':
            lp_x = fd['val_trend'] + mx.predict(X_val)
            lp_l = fd['val_trend'] + ml.predict(X_val)
            lp_c = fd['val_trend'] + mc.predict(X_val)
            oof_xgb.extend(lp_x); oof_lgb.extend(lp_l); oof_cat.extend(lp_c)
            oof_y.extend(fd['y_val_rev'] + fd['val_trend'])
            oof_y_rev_true.extend(fd['y_val_true'])
            mae_x = mean_absolute_error(fd['y_val_true'], np.expm1(lp_x))
            mae_l = mean_absolute_error(fd['y_val_true'], np.expm1(lp_l))
            mae_c = mean_absolute_error(fd['y_val_true'], np.expm1(lp_c))
            print(f"  Revenue MAE → XGB:{mae_x:,.0f}  LGB:{mae_l:,.0f}  CAT:{mae_c:,.0f}")
        else:
            oof_xgb.extend(mx.predict(X_val))
            oof_lgb.extend(ml.predict(X_val))
            oof_cat.extend(mc.predict(X_val))
            oof_y.extend(y_val)
            if target_name == 'COGS_log':
                oof_y_cog_true.extend(fd['y_val_cog_true'])

    oof_dates_all.extend(fd['val_dates'])

# ==============================================================================
# 9. META MODELS — HuberRegressor blend OOF predictions
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 6 — Meta Models (HuberRegressor)")
print("=" * 65)

huber_rev  = {'epsilon': [1.2, 1.25, 1.35, 1.4, 1.45, 1.5],  'alpha': [1e-4, 1e-3, 1e-2, 0.1]}
huber_cogs = {'epsilon': [1.01, 1.05, 1.1, 1.2, 1.35],        'alpha': [1e-4, 1e-3, 1e-2, 0.1]}

def fit_meta(X, y, grid):
    gs = GridSearchCV(HuberRegressor(fit_intercept=False), grid,
                      cv=5, scoring='neg_mean_absolute_error')
    gs.fit(X, y)
    return gs.best_estimator_

X_meta_rev = np.column_stack([oof_rev_xgb, oof_rev_lgb, oof_rev_cat])
X_meta_rat = np.column_stack([oof_rat_xgb, oof_rat_lgb, oof_rat_cat])
X_meta_cog = np.column_stack([oof_cog_xgb, oof_cog_lgb, oof_cog_cat])

meta_rev_model = fit_meta(X_meta_rev, oof_y_rev_log, huber_rev)
meta_rat_model = fit_meta(X_meta_rat, oof_y_rat,     huber_cogs)
meta_cog_model = fit_meta(X_meta_cog, oof_y_cog_log, huber_cogs)

print(f"Revenue meta  (XGB/LGB/CAT): {meta_rev_model.coef_.round(4)}")
print(f"Ratio  meta   (XGB/LGB/CAT): {meta_rat_model.coef_.round(4)}")
print(f"COGSlog meta  (XGB/LGB/CAT): {meta_cog_model.coef_.round(4)}")

oof_rev_meta = np.expm1(meta_rev_model.predict(X_meta_rev))
print(f"OOF Revenue MAE (meta blend): {mean_absolute_error(oof_y_rev_true, oof_rev_meta):,.0f}")

# ==============================================================================
# 10. TỐI ƯU ALPHA BLEND COGS
#     COGS_indep = expm1(log-COGS model) — dự báo vốn bỏ ra đầu ngày
#     COGS_dep   = Revenue × COGS_Ratio  — dự báo phụ thuộc doanh thu
#     COGS_final = alpha × indep + (1-alpha) × dep
#     alpha được tối ưu bằng scipy.minimize_scalar trên OOF
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 7 — COGS Blend Alpha Optimization (scipy)")
print("=" * 65)

oof_cogs_indep = np.expm1(meta_cog_model.predict(X_meta_cog)).clip(0)
oof_cogs_ratio = meta_rat_model.predict(X_meta_rat).clip(0, 0.99)
oof_cogs_dep   = (oof_rev_meta * oof_cogs_ratio).clip(0)
oof_y_cogs     = np.array(oof_y_cog_true)

def cogs_blend_mae(alpha):
    blend = alpha * oof_cogs_indep + (1 - alpha) * oof_cogs_dep
    blend = np.minimum(blend, oof_rev_meta * 0.99).clip(0)
    return mean_absolute_error(oof_y_cogs, blend)

result     = minimize_scalar(cogs_blend_mae, bounds=(0.0, 1.0), method='bounded')
ALPHA_COGS = float(result.x)

print(f"Alpha tối ưu : {ALPHA_COGS:.4f}  (0=ratio-only  →  1=log-COGS-only)")
print(f"  COGS MAE indep-only : {mean_absolute_error(oof_y_cogs, oof_cogs_indep):,.0f}")
print(f"  COGS MAE ratio-only : {mean_absolute_error(oof_y_cogs, oof_cogs_dep):,.0f}")
print(f"  COGS MAE blend      : {cogs_blend_mae(ALPHA_COGS):,.0f}")

# ==============================================================================
# 11. SAMPLE WEIGHTS — high-error year nhận weight cao hơn
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 8 — Sample Weights (OOF Year-Level MAE)")
print("=" * 65)

oof_df = pd.DataFrame({
    'Date'  : pd.to_datetime(oof_dates_all),
    'actual': np.array(oof_y_rev_true),
    'pred'  : oof_rev_meta,
})
oof_df['year']    = oof_df['Date'].dt.year
oof_df['abs_err'] = np.abs(oof_df['actual'] - oof_df['pred'])
year_mae    = oof_df.groupby('year')['abs_err'].mean()
mean_mae    = year_mae.mean()
year_weight = ((year_mae / mean_mae) ** 0.5).clip(0.5, 2.0)

print("Per-year OOF MAE → sample weight:")
for yr, (mae, w) in pd.concat([year_mae, year_weight], axis=1).rename(
        columns={0: 'mae', 1: 'weight'}).iterrows():
    print(f"  {yr}: MAE={mae:,.0f}  weight={w:.3f}")

sample_weight_train = df_sales['Date'].dt.year.map(year_weight).fillna(1.0).values

# ==============================================================================
# 12. RETRAIN FULL DATA VỚI SAMPLE WEIGHTS
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 9 — Retrain Full Data (3 targets × 3 models × sample_weight)")
print("=" * 65)

final_prophet  = fit_prophet(df_sales['Date'], df_sales['Revenue_log'])
comp_full      = prophet_components(final_prophet, df_sales['Date'])
train_res_full = df_sales['Revenue_log'].values - comp_full['yhat']

def make_X_full(static_df, comp):
    sf = static_df.copy()
    for col in PROPHET_COLS:
        sf[col] = comp[col]
    sf = add_interactions(sf)
    sf[ALL_FEAT_COLS] = sf[ALL_FEAT_COLS].fillna(-1)
    return sf[ALL_FEAT_COLS].values

X_full = make_X_full(train_static, comp_full)
sw     = sample_weight_train

XGB_FULL = {**XGB_BEST, 'n_estimators': 600}
LGB_FULL = {**LGB_BEST, 'n_estimators': 600}
CAT_FULL = {**CAT_BEST, 'iterations':   600}

def train_trio(X, y, sw, xgb_p, lgb_p, cat_p):
    mx = xgb.XGBRegressor(**xgb_p)
    mx.fit(X, y, sample_weight=sw, verbose=False)
    ml = lgb.LGBMRegressor(**lgb_p)
    ml.fit(X, y, sample_weight=sw)
    mc = CatBoostRegressor(**cat_p)
    mc.fit(X, y, sample_weight=sw, verbose=False)
    return mx, ml, mc

f_rev_xgb, f_rev_lgb, f_rev_cat = train_trio(X_full, train_res_full,               sw, XGB_FULL, LGB_FULL, CAT_FULL)
f_rat_xgb, f_rat_lgb, f_rat_cat = train_trio(X_full, df_sales['COGS_Ratio'].values, sw, XGB_FULL, LGB_FULL, CAT_FULL)
f_cog_xgb, f_cog_lgb, f_cog_cat = train_trio(X_full, df_sales['COGS_log'].values,   sw, XGB_FULL, LGB_FULL, CAT_FULL)

print("Full retrain hoàn tất")

# ==============================================================================
# 13. FORECAST TEST SET + COGS BLEND
# ==============================================================================
print("\n" + "=" * 65)
print("STEP 10 — Forecast + COGS Blend")
print("=" * 65)

comp_test  = prophet_components(final_prophet, df_sub['Date'])
X_test     = make_X_full(test_static, comp_test)
test_trend = comp_test['yhat']

def predict_trio(models, X, trend=None):
    mx, ml, mc = models
    px = mx.predict(X); pl = ml.predict(X); pc = mc.predict(X)
    if trend is not None:
        px += trend; pl += trend; pc += trend
    return np.column_stack([px, pl, pc])

# Revenue
X_meta_rev_test = predict_trio((f_rev_xgb, f_rev_lgb, f_rev_cat), X_test, test_trend)
pred_rev        = np.expm1(meta_rev_model.predict(X_meta_rev_test)).clip(0)

# COGS ratio → phụ thuộc Revenue
X_meta_rat_test = predict_trio((f_rat_xgb, f_rat_lgb, f_rat_cat), X_test)
pred_ratio      = meta_rat_model.predict(X_meta_rat_test).clip(0, 0.99)
cogs_dep        = (pred_rev * pred_ratio).clip(0)

# COGS log → độc lập
X_meta_cog_test = predict_trio((f_cog_xgb, f_cog_lgb, f_cog_cat), X_test)
cogs_indep      = np.expm1(meta_cog_model.predict(X_meta_cog_test)).clip(0)

# Blend + enforce: 0 < COGS < Revenue
pred_cogs = ALPHA_COGS * cogs_indep + (1 - ALPHA_COGS) * cogs_dep
pred_cogs = np.minimum(pred_cogs, pred_rev * 0.99).clip(0)

print(f"Revenue: mean={pred_rev.mean():,.0f}  min={pred_rev.min():,.0f}  max={pred_rev.max():,.0f}")
print(f"COGS   : mean={pred_cogs.mean():,.0f}  COGS/Revenue={pred_cogs.mean()/pred_rev.mean():.4f}")

# ==============================================================================
# 14. XUẤT FILE SUBMISSION
# ==============================================================================
test_dates = df_sub['Date'].tolist()
final_sub = pd.DataFrame({
    'Date'   : [d.strftime('%Y-%m-%d') for d in test_dates],
    'Revenue': np.round(pred_rev,  2),
    'COGS'   : np.round(pred_cogs, 2),
})
final_sub.to_csv(OUTPUT_FILE, index=False)

assert (final_sub['Revenue'] >= 0).all(),   "Revenue phải >= 0"
assert (final_sub['COGS']    >= 0).all(),   "COGS phải >= 0"
assert (final_sub['COGS']    <= final_sub['Revenue']).all(), "COGS phải <= Revenue"

print(f"\n {OUTPUT_FILE} — {len(final_sub)} rows")
print(final_sub.head(5).to_string(index=False))

# Colab download (bỏ qua nếu chạy local)
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except Exception:
    print("(Local run — file saved to disk)")


STEP 1 — Tải dữ liệu


FileNotFoundError: [Errno 2] No such file or directory: '../dataset/sample_submission.csv'

In [ ]:
import shap
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap, Normalize
import numpy as np

# ── 1. Train model trên fold cuối ───────────────────────────
fd_last = fold_cache[-1]
X_shap  = fd_last['X_val']
y_shap  = fd_last['y_val_rev']

lgb_shap = lgb.LGBMRegressor(**LGB_BEST)
lgb_shap.fit(
    fd_last['X_tr'], fd_last['y_tr_rev'],
    eval_set=[(X_shap, y_shap)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

# ── 2. Tính SHAP values ──────────────────────────────────────
explainer   = shap.TreeExplainer(lgb_shap)
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list):
    shap_values = shap_values[0]

mean_abs_shap = np.abs(shap_values).mean(axis=0)
sorted_idx    = np.argsort(mean_abs_shap)
TOP_N         = 15
top_idx       = sorted_idx[-TOP_N:]

# ── Style constants ──────────────────────────────────────────
CMAP   = LinearSegmentedColormap.from_list('shap', ['#1a78c2', '#cccccc', '#cc3030'])
ACCENT = '#1a3a5c'
BG     = '#f5f7fa'

# ════════════════════════════════════════════════════════════
# Plot A: Mean |SHAP| bar chart
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

vals  = mean_abs_shap[top_idx]
names = [ALL_FEAT_COLS[i] for i in top_idx]

# Gradient màu theo giá trị
norm_vals = vals / vals.max()
bar_colors = plt.cm.Blues(0.35 + 0.6 * norm_vals)

bars = ax.barh(names, vals, color=bar_colors, edgecolor='white', height=0.7)

# Highlight top-3 bars
for bar in bars[-3:]:
    bar.set_color(ACCENT)

# Label giá trị
x_offset = vals.max() * 0.012
for bar, v in zip(bars, vals):
    ax.text(
        v + x_offset,
        bar.get_y() + bar.get_height() / 2,
        f'{v:.3f}',
        va='center', ha='left', fontsize=8, color='#333333'
    )

ax.set_xlim(0, vals.max() * 1.15)   # tránh label bị crop
ax.set_xlabel('Mean |SHAP value|  (log-revenue scale)', fontsize=10)
ax.set_title('Feature Importance — Mean |SHAP|', fontsize=12,
             fontweight='bold', color=ACCENT, pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ shap_bar.png")

# ════════════════════════════════════════════════════════════
# Plot B: Beeswarm (tự vẽ — compatible với mọi phiên bản SHAP)
# ════════════════════════════════════════════════════════════
rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

for rank, fi in enumerate(top_idx):
    sv = shap_values[:, fi]

    # Normalise feature values 0→1 (sửa ptp() deprecated)
    fv  = X_shap[:, fi].astype(float)
    fv_range = fv.max() - fv.min()
    fv_norm  = (fv - fv.min()) / (fv_range + 1e-9)

    y_jitter = rng.uniform(-0.35, 0.35, len(sv))
    order    = np.argsort(np.abs(sv))   # điểm lớn vẽ sau (nổi trên cùng)

    ax.scatter(
        sv[order], rank + y_jitter[order],
        c=fv_norm[order], cmap=CMAP, vmin=0, vmax=1,
        s=8, alpha=0.65, linewidths=0,
    )

ax.set_yticks(range(TOP_N))
ax.set_yticklabels([ALL_FEAT_COLS[i] for i in top_idx], fontsize=9)
ax.axvline(0, color='#888888', linewidth=0.9, zorder=0)
ax.set_xlabel('SHAP value  (impact on log Revenue)', fontsize=10)
ax.set_title('SHAP Beeswarm — Direction & Magnitude of Impact',
             fontsize=12, fontweight='bold', color=ACCENT, pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.grid(axis='x', alpha=0.25, linestyle='--', zorder=0)

# Colorbar dùng ScalarMappable riêng (không gắn vào scatter)
sm   = cm.ScalarMappable(cmap=CMAP, norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.45, aspect=20, pad=0.02)
cbar.set_label('Feature value (normalized)', fontsize=9)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(['Low', 'Mid', 'High'])
cbar.outline.set_visible(False)

plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ shap_beeswarm.png")

# ════════════════════════════════════════════════════════════
# Plot C: Dependence plot (top-1 feature, tô màu theo top-2)
# ════════════════════════════════════════════════════════════
top1 = int(sorted_idx[-1])
top2 = int(sorted_idx[-2])

fig, ax = plt.subplots(figsize=(7, 4.5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

x_v = X_shap[:, top1].astype(float)
y_v = shap_values[:, top1]
c_v = X_shap[:, top2].astype(float)
c_v_norm = (c_v - c_v.min()) / ((c_v.max() - c_v.min()) + 1e-9)  # sửa ptp()

sc = ax.scatter(
    x_v, y_v,
    c=c_v_norm, cmap=CMAP, vmin=0, vmax=1,
    s=20, alpha=0.65, linewidths=0.3, edgecolors='white',
)

# Đường trend (robust: dùng np.polyfit trên data đã sort)
sort_mask = np.argsort(x_v)
z    = np.polyfit(x_v, y_v, 1)
xlin = np.linspace(x_v.min(), x_v.max(), 300)
ax.plot(xlin, np.polyval(z, xlin),
        color=ACCENT, lw=1.8, ls='--', label='Linear trend', zorder=5)
ax.axhline(0, color='#888888', lw=0.8, zorder=0)

ax.set_xlabel(ALL_FEAT_COLS[top1], fontsize=10)
ax.set_ylabel(f'SHAP value  [{ALL_FEAT_COLS[top1]}]', fontsize=10)
ax.set_title(
    f'Dependence Plot: {ALL_FEAT_COLS[top1]}\n'
    f'(color = {ALL_FEAT_COLS[top2]})',
    fontsize=11, fontweight='bold', color=ACCENT, pad=12,
)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2, linestyle='--', zorder=0)
ax.legend(fontsize=8, framealpha=0)

cbar2 = fig.colorbar(sc, ax=ax, shrink=0.8, aspect=18)
cbar2.set_label(ALL_FEAT_COLS[top2], fontsize=8)
cbar2.outline.set_visible(False)

plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ shap_dependence.png")